In [1]:
# IMPORTING NECESSARY LIBRARY
# imports the pandas library(used to manipulate and load tabular data), numpy for scientific computing, tensorflow for neural_network

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models

In [3]:
# LOADING DATA
X_train = pd.read_parquet("x_train_selected.parquet")
X_test  = pd.read_parquet("x_test_selected.parquet")
y_test  = pd.read_parquet("y_test.parquet").values.ravel()
y_train = pd.read_parquet("y_train.parquet").values.ravel()

# keep only normal samples
X_train_normal = X_train[y_train == 0]

# Convert labels to binary: 0 = normal, 1 = anomaly
y_test_bin = (y_test != 0).astype(int)
expected_anomalies = int(y_test_bin.sum())

In [5]:
#SCALE FEATURES
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_normal)
X_test_scaled  = scaler.transform(X_test)

In [7]:
# ISOLATION FOREST

# Initializing the model
# n_estimators=200 → number of trees (more = more stable)
# contamination="auto"→ expected % of anomalies
# random_state=42 → reproducibility
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(X_train_scaled) # To train model only on feature data

train_iso_scores = -iso.decision_function(X_train_scaled)  # higher = more anomalous
test_iso_scores = -iso.decision_function(X_test_scaled)

# Choose threshold manually to match expected anomalies
threshold_iso = np.percentile(train_iso_scores, 95) # top N anomalies
iso_pred_bin = (test_iso_scores >= threshold_iso).astype(int)

print("Isolation Forest")
print("ROC-AUC:", roc_auc_score(y_test_bin, test_iso_scores))
print(classification_report(y_test_bin, iso_pred_bin))
fp_iso = ((iso_pred_bin == 1) & (y_test_bin == 0)).sum()
print("False Positives:", fp_iso, "\n")

Isolation Forest
ROC-AUC: 0.8387143320904898
              precision    recall  f1-score   support

           0       0.45      0.95      0.61     49601
           1       0.80      0.15      0.26     67298

    accuracy                           0.49    116899
   macro avg       0.63      0.55      0.43    116899
weighted avg       0.65      0.49      0.41    116899

False Positives: 2506 



In [ ]:
# ONE-CLASS SVM

# kernel="rbf" → non-linear boundary
# gamma="scale" → automatic kernel scaling
# nu=0.05 → max fraction of anomalies
# It works well for complex distributions but slower
#ocsvm = OneClassSVM(
    #kernel="linear",
    #gamma="scale",
    #nu=0.05
#)

# Fits boundary around normal data
#ocsvm.fit(X_train_scaled)

# Distance from leared boundary ( more negative -> more anomalous)
#svm_scores = -ocsvm.decision_function(X_test_scaled)
#print(f"One-Class SVM score: {svm_scores}\n")

# Setting Threshold
#svm_threshold = np.percentile(svm_scores, 95)
#print(f"One-Class SVM threshold: {svm_threshold}\n")

# Converts predictions to anomaly labels
#svm_pred = ocsvm.predict(X_test_scaled)
#print(f"One-Class SVM predictions: {svm_pred}\n")

# Convert prediction to binary format
#y_pred = (svm_scores >= svm_threshold).astype(int)

# ONE-CLASS SVM - ROC_AUC and FALSEPOSITIVE ANALYSIS
#print("=== One-Class SVM ===")
#print("ROC-AUC:", roc_auc_score(y_test_bin, svm_scores))
#print(classification_report(y_test_bin, svm_pred))
#fp = ((svm_pred==1) & (y_test_bin==0)).sum()
#print("False Positives:", fp, "\n")

In [ ]:
# AUTOENCODER
# Gets number of input features because neural networks must know input size.
input_dim = X_train_scaled.shape[1]

# Creates a sequential neural network
# Encoder compresses data and force model to learn important structure
# Decoder recontructs original input
autoencoder = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(input_dim, activation="linear")
])

# Configures training (Adam → fast convergence,MSE → reconstruction accuracy)
autoencoder.compile(optimizer="adam", loss="mse")

# Trains model to reconstruct normal data
autoencoder.fit(
    X_train_scaled, X_train_scaled,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

# TRAIN reconstruction
train_pred = autoencoder.predict(X_train_scaled)
train_error = np.mean((X_train_scaled - train_pred) ** 2, axis=1)

# TEST reconstruction
test_pred = autoencoder.predict(X_test_scaled)
test_error = np.mean((X_test_scaled - test_pred) ** 2, axis=1)
threshold_ae = np.percentile(train_error, 95)
ae_pred_bin = (test_error >= threshold_ae).astype(int)

print("Autoencoder")
print("ROC-AUC:", roc_auc_score(y_test_bin, test_error))
print(classification_report(y_test_bin, ae_pred_bin))
fp_ae = ((ae_pred_bin == 1) & (y_test_bin == 0)).sum()
print("False Positives:", fp_ae, "\n")

Epoch 1/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.2640 - val_loss: 0.0597
Epoch 2/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0989 - val_loss: 0.0349
Epoch 3/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0676 - val_loss: 0.0256
Epoch 4/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0498 - val_loss: 0.0233
Epoch 5/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0393 - val_loss: 0.0178
Epoch 6/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0397 - val_loss: 0.0225
Epoch 7/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0284 - val_loss: 0.0285
Epoch 8/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0278 - val_loss: 0.0145
Epoch 9/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0245 - val_loss: 0.0114
Epoch 10/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0248 - val_loss: 0.0111
Epoch 11/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0218 - val_loss: 0.0321
Epoch 12/50
698/698 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step

In [13]:
# SUMMARY

print("Summary of false positives:")
print(f"Isolation Forest: {fp_iso}")
#print(f"One-Class SVM: {fp_svm}")
print(f"Autoencoder: {fp_ae}")

Summary of false positives:
Isolation Forest: 2506
Autoencoder: 2491


In [21]:
# SAVE ISOLATION FOREST
import pickle

iso_artifacts = {
    "scaler": scaler,                    # fitted on normal training data
    "model": iso,                        # trained Isolation Forest
    "threshold": threshold_iso,          # decision threshold
    "feature_columns": X_train.columns.tolist()
}

with open("isolation_forest.pkl", "wb") as f:
    pickle.dump(iso_artifacts, f)

print("Isolation Forest saved successfully.")


Isolation Forest saved successfully.


In [23]:
# LOAD ISOLATION FOREST
with open("isolation_forest.pkl", "rb") as f:
    iso_artifacts = pickle.load(f)

scaler = iso_artifacts["scaler"]
iso = iso_artifacts["model"]
threshold_iso = iso_artifacts["threshold"]

In [27]:
# SAVE AUTOENCODER

autoencoder.save("autoencoder_model.keras")
print("Autoencoder model saved.")


Autoencoder model saved.


In [29]:
# SAVE AUTOENCODER THRESHOLD

ae_artifacts = {
    "threshold": threshold_ae
}

with open("autoencoder_threshold.pkl", "wb") as f:
    pickle.dump(ae_artifacts, f)

print("Autoencoder threshold saved.")

Autoencoder threshold saved.


In [33]:
# LOAD AUTOENCODER

from tensorflow.keras.models import load_model

autoencoder = load_model("autoencoder_model.keras")

with open("autoencoder_threshold.pkl", "rb") as f:
    ae_artifacts = pickle.load(f)

threshold_ae = ae_artifacts["threshold"]
